# Wildfire Susceptibility Model Evaluation

This notebook evaluates the performance of a Random Forest wildfire susceptibility model developed for Butte County, California.

Objectives:

- Assess model performance
- Examine cross-validation stability
- Evaluate classification accuracy
- Identify important wildfire predictors

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from IPython.display import display

import rasterio

from pathlib import Path

In [ ]:
PROJECT_ROOT = Path("..")

MODEL_EVAL_DIR = (
    PROJECT_ROOT /
    "outputs" /
    "model_evaluation"
)

FEATURE_IMPORTANCE_DIR = (
    PROJECT_ROOT /
    "outputs" /
    "feature_importance"
)

METRICS_PATH = (
    MODEL_EVAL_DIR /
    "metrics.csv"
)

CV_PATH = (
    MODEL_EVAL_DIR /
    "cv_scores.csv"
)

CLASSIFICATION_REPORT_PATH = (
    MODEL_EVAL_DIR /
    "classification_report.csv"
)

ROC_PATH = (
    MODEL_EVAL_DIR /
    "roc_curve.png"
)

CONFUSION_PATH = (
    MODEL_EVAL_DIR /
    "confusion_matrix.png"
)

FEATURE_IMPORTANCE_PATH = (
    FEATURE_IMPORTANCE_DIR /
    "feature_importance.csv"
)

## metrics

The Random Forest model achieved excellent predictive performance with an ROC-AUC of 0.994 and overall accuracy exceeding 96%.

In [ ]:
metrics = pd.read_csv(
    METRICS_PATH
)

metrics

## Cross Validation Table

In [ ]:
cv_scores = pd.read_csv(
    CV_PATH
)

cv_scores

## Cross-Validation ROC-AUC Plot.

In [ ]:
mean_auc = cv_scores["ROC_AUC"].mean()

plt.figure(figsize=(8,5))

plt.bar(
    cv_scores["Fold"],
    cv_scores["ROC_AUC"]
)

plt.axhline(
    mean_auc,
    linestyle="--",
    linewidth=2,
    label=f"Mean = {mean_auc:.4f}"
)

plt.ylim(
    0.985,
    0.990
)

plt.xlabel(
    "Fold"
)

plt.ylabel(
    "ROC-AUC"
)

plt.title(
    "Cross-Validation ROC-AUC Scores"
)

plt.legend()

plt.tight_layout()

output_path = (
    PROJECT_ROOT /
    "outputs" /
    "figures" /
    "cross_validation_scores.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

print(
    f"Saved: {output_path}"
)

plt.show()

### Cross-Validation Interpretation

Five-fold cross-validation produced highly consistent ROC-AUC scores ranging from 0.9872 to 0.9888, with a mean ROC-AUC of 0.9881.

The small variation among folds indicates that the Random Forest model is stable and generalizes well across different subsets of the data. The consistently high ROC-AUC values demonstrate strong discrimination between wildfire and non-wildfire observations and suggest a low risk of model overfitting.

## Classification Report

In [ ]:
classification_report = pd.read_csv(
    CLASSIFICATION_REPORT_PATH
)

classification_report

### Classification Report Interpretation

The Random Forest model achieved strong and balanced classification performance for both wildfire and non-wildfire classes.

For burned locations (Class 1), the model achieved:

- Precision: 95.8%
- Recall: 96.4%
- F1-Score: 96.1%

For unburned locations (Class 0), the model achieved:

- Precision: 96.4%
- Recall: 95.8%
- F1-Score: 96.1%

The similarity between class-specific metrics indicates that the model performs consistently across both classes and is not strongly biased toward predicting either burned or unburned areas.

The weighted F1-score of 96.1% demonstrates excellent overall classification performance.

## ROC-Curve

In [ ]:
display(
    Image.open(
        ROC_PATH
    )
)

### ROC Curve Interpretation

The Random Forest model achieved an ROC-AUC score of 0.994, indicating excellent discrimination between wildfire and non-wildfire locations. The curve remains close to the upper-left corner of the ROC space, reflecting strong predictive performance.

## Confusion Metrix

In [ ]:
display(
    Image.open(
        CONFUSION_PATH
    )
)

In [ ]:
confusion_summary = pd.DataFrame(
    {
        "Metric": [
            "True Negatives",
            "False Positives",
            "False Negatives",
            "True Positives"
        ],
        "Count": [
            19163,
            837,
            719,
            19281
        ]
    }
)

confusion_summary

### Confusion Matrix Interpretation

The confusion matrix demonstrates strong classification performance for both wildfire and non-wildfire classes.

- True Negatives (TN): 19,163
- False Positives (FP): 837
- False Negatives (FN): 719
- True Positives (TP): 19,281

Only 1,556 of 40,000 testing samples were misclassified, resulting in an overall accuracy of 96.1%.

The balanced distribution of false positives and false negatives indicates that the model is not strongly biased toward either class.

## Feature Importance

In [ ]:
feature_importance = pd.read_csv(
    FEATURE_IMPORTANCE_PATH
)

feature_importance.head()

In [ ]:
feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=True
)

plt.figure(figsize=(8,6))

plt.barh(
    feature_importance["Feature"],
    feature_importance["Importance"]
)

plt.xlabel(
    "Importance"
)

plt.ylabel(
    "Feature"
)

plt.title(
    "Random Forest Feature Importance"
)

plt.tight_layout()

output_path = (
    PROJECT_ROOT /
    "outputs" /
    "figures" /
    "feature_importance_notebook.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

print(
    f"Saved: {output_path}"
)

plt.show()

### Feature Importance Interpretation

The feature importance analysis indicates that precipitation and elevation (DEM) were the most influential predictors of wildfire susceptibility in Butte County.

Land cover and slope also contributed substantially to model performance, highlighting the importance of vegetation characteristics and terrain conditions in wildfire occurrence. Human-related factors, such as distance to settlements, provided additional predictive value but were less influential than environmental variables.

Overall, the results suggest that wildfire susceptibility in the study area is primarily driven by a combination of climatic conditions, topography, and land cover patterns, with human influences playing a secondary role.

## Key Findings

1. The Random Forest model achieved excellent predictive performance with an ROC-AUC of 0.994 and an overall accuracy of 96.1%.

2. Five-fold cross-validation produced highly consistent ROC-AUC scores (mean = 0.988), indicating strong model stability and low risk of overfitting.

3. Classification metrics demonstrated balanced performance for both wildfire and non-wildfire classes.

4. The confusion matrix showed only 1,556 misclassified samples out of 40,000 testing observations.

5. Precipitation, elevation, land cover, and slope were identified as the most important predictors of wildfire susceptibility.

6. The resulting susceptibility model provides a robust foundation for wildfire hazard assessment and risk mapping in Butte County, California.